In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q gdown ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.1 MB/s eta 0:00:00a 0:00:01


In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [4]:
!gdown 18umLJ4x8uHAW3bB9sQWit_hBm27x2fXi -O taco_yolo.zip
!ls -la taco_yolo.zip

Downloading...
From (original): https://drive.google.com/uc?id=18umLJ4x8uHAW3bB9sQWit_hBm27x2fXi
From (redirected): https://drive.google.com/uc?id=18umLJ4x8uHAW3bB9sQWit_hBm27x2fXi&confirm=t&uuid=c4004da3-28e9-48f8-b08e-70217f4d2629
To: /kaggle/working/taco_yolo.zip
100%|███████████████████████████████████████| 2.62G/2.62G [00:19<00:00, 132MB/s]
-rw-r--r-- 1 root root 2618033595 Aug  3 21:37 taco_yolo.zip


In [5]:
!unzip -o -q taco_yolo.zip -d /kaggle/working/
!find /kaggle/working -iname "data.yaml"

/kaggle/working/taco_yolo/data.yaml


In [6]:
import os
import glob
import yaml

# 1. Updated mapping to 12 English Macro-Categories (including WEEE & Hazardous)
NAME_TO_MACRO_IDX = {
    'Aluminium foil': 3,              # Metal_and_Cans
    'Battery': 4,                     # WEEE_and_Electronics ⚡
    'Aluminium blister pack': 5,      # Hazardous_and_Toxic ⚠️ (medicine)
    'Carded blister pack': 5,         # Hazardous_and_Toxic ⚠️ (medicine)
    'Other plastic bottle': 0,        # Plastic_Bottles_and_Containers
    'Clear plastic bottle': 0,        # Plastic_Bottles_and_Containers
    'Glass bottle': 6,                # Glass
    'Plastic bottle cap': 2,          # Caps_and_Lids
    'Metal bottle cap': 2,            # Caps_and_Lids
    'Broken glass': 6,                # Glass
    'Food Can': 3,                    # Metal_and_Cans
    'Aerosol': 5,                     # Hazardous_and_Toxic ⚠️ (spray cans)
    'Drink can': 3,                   # Metal_and_Cans
    'Toilet tube': 7,                 # Paper_and_Cardboard
    'Other carton': 7,                # Paper_and_Cardboard
    'Egg carton': 7,                  # Paper_and_Cardboard
    'Drink carton': 7,                # Paper_and_Cardboard
    'Corrugated carton': 7,           # Paper_and_Cardboard
    'Meal carton': 7,                 # Paper_and_Cardboard
    'Pizza box': 7,                   # Paper_and_Cardboard
    'Paper cup': 7,                   # Paper_and_Cardboard
    'Disposable plastic cup': 1,      # Plastic_Films_and_Wrappers
    'Foam cup': 8,                    # Styrofoam
    'Glass cup': 6,                   # Glass
    'Other plastic cup': 1,           # Plastic_Films_and_Wrappers
    'Food waste': 10,                 # Organic_Waste 🍏
    'Glass jar': 6,                   # Glass
    'Plastic lid': 2,                 # Caps_and_Lids
    'Metal lid': 2,                   # Caps_and_Lids
    'Other plastic': 1,               # Plastic_Films_and_Wrappers
    'Magazine paper': 7,              # Paper_and_Cardboard
    'Tissues': 7,                     # Paper_and_Cardboard
    'Wrapping paper': 7,              # Paper_and_Cardboard
    'Normal paper': 7,                # Paper_and_Cardboard
    'Paper bag': 7,                   # Paper_and_Cardboard
    'Plastified paper bag': 7,        # Paper_and_Cardboard
    'Plastic film': 1,                # Plastic_Films_and_Wrappers
    'Six pack rings': 1,              # Plastic_Films_and_Wrappers
    'Garbage bag': 1,                 # Plastic_Films_and_Wrappers
    'Other plastic wrapper': 1,       # Plastic_Films_and_Wrappers
    'Single-use carrier bag': 1,      # Plastic_Films_and_Wrappers
    'Polypropylene bag': 1,           # Plastic_Films_and_Wrappers
    'Crisp packet': 1,                # Plastic_Films_and_Wrappers
    'Spread tub': 0,                  # Plastic_Bottles_and_Containers
    'Tupperware': 0,                  # Plastic_Bottles_and_Containers
    'Disposable food container': 0,   # Plastic_Bottles_and_Containers
    'Foam food container': 8,         # Styrofoam
    'Other plastic container': 0,     # Plastic_Bottles_and_Containers
    'Plastic glooves': 1,             # Plastic_Films_and_Wrappers
    'Plastic utensils': 1,            # Plastic_Films_and_Wrappers
    'Pop tab': 2,                     # Caps_and_Lids
    'Rope & strings': 4,              # WEEE_and_Electronics ⚡ (cables & strings)
    'Scrap metal': 3,                 # Metal_and_Cans
    'Shoe': 11,                       # Other_Trash
    'Squeezable tube': 1,             # Plastic_Films_and_Wrappers
    'Plastic straw': 1,               # Plastic_Films_and_Wrappers
    'Paper straw': 7,                 # Paper_and_Cardboard
    'Styrofoam piece': 8,             # Styrofoam
    'Unlabeled litter': 11,           # Other_Trash
    'Cigarette': 9                    # Cigarette_Butts 🚬
}

macro_names = {
    0: 'Plastic_Bottles_and_Containers',
    1: 'Plastic_Films_and_Wrappers',
    2: 'Caps_and_Lids',
    3: 'Metal_and_Cans',
    4: 'WEEE_and_Electronics',
    5: 'Hazardous_and_Toxic',
    6: 'Glass',
    7: 'Paper_and_Cardboard',
    8: 'Styrofoam',
    9: 'Cigarette_Butts',
    10: 'Organic_Waste',
    11: 'Other_Trash'
}

# 2. Read original data.yaml to fetch class indices
data_yaml_path = "/kaggle/working/taco_yolo/data.yaml"
with open(data_yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

original_names = cfg["names"]

# 3. Create ID mapping: OLD_ID (0-59) -> NEW_MACRO_ID (0-11)
id_to_macro = {}
for old_id, original_name in original_names.items():
    macro_idx = NAME_TO_MACRO_IDX.get(original_name, 11)
    id_to_macro[int(old_id)] = macro_idx

# 4. Rewrite all label files (.txt)
def remap_label_file(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        old_class_id = int(parts[0])
        new_class_id = id_to_macro.get(old_class_id, 11)
        new_lines.append(f"{new_class_id} " + " ".join(parts[1:]) + "\n")
        
    with open(file_path, 'w') as f:
        f.writelines(new_lines)

label_files = glob.glob("/kaggle/working/taco_yolo/labels/**/*.txt", recursive=True)
for file_path in label_files:
    remap_label_file(file_path)

print(f"✅ Converted {len(label_files)} label files to 12 English Macro-Categories!")

# 5. Update data.yaml configuration
cfg["path"] = "/kaggle/working/taco_yolo"
cfg["nc"] = len(macro_names)
cfg["names"] = macro_names

with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("\nUpdated data.yaml with 12 English classes:")
print(cfg)

✅ Converted 1500 label files to 12 English Macro-Categories!

Updated data.yaml with 12 English classes:
{'path': '/kaggle/working/taco_yolo', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'Plastic_Bottles_and_Containers', 1: 'Plastic_Films_and_Wrappers', 2: 'Caps_and_Lids', 3: 'Metal_and_Cans', 4: 'WEEE_and_Electronics', 5: 'Hazardous_and_Toxic', 6: 'Glass', 7: 'Paper_and_Cardboard', 8: 'Styrofoam', 9: 'Cigarette_Butts', 10: 'Organic_Waste', 11: 'Other_Trash'}, 'nc': 12}


In [7]:
import yaml

data_yaml_path = "/kaggle/working/taco_yolo/data.yaml"
with open(data_yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = "/kaggle/working/taco_yolo"

with open(data_yaml_path, "w") as f:
    yaml.dump(cfg, f)

print(cfg)

{'path': '/kaggle/working/taco_yolo', 'train': 'images/train', 'val': 'images/val', 'names': {0: 'Plastic_Bottles_and_Containers', 1: 'Plastic_Films_and_Wrappers', 2: 'Caps_and_Lids', 3: 'Metal_and_Cans', 4: 'WEEE_and_Electronics', 5: 'Hazardous_and_Toxic', 6: 'Glass', 7: 'Paper_and_Cardboard', 8: 'Styrofoam', 9: 'Cigarette_Butts', 10: 'Organic_Waste', 11: 'Other_Trash'}, 'nc': 12}


In [8]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/kaggle/working/taco_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/kaggle/working/runs_taco",
    name="yolo26n_taco",
)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/taco_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud

In [9]:
model.export(format="onnx", imgsz=640)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,377,176 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/kaggle/working/runs_taco/yolo26n_taco/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 403ms
Prepared 2 packages in 366ms
Installed 2 packages in 13ms
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success ✅ 3.

'/kaggle/working/runs_taco/yolo26n_taco/weights/best.onnx'

In [10]:
import shutil

save_dir = str(results.save_dir)
print(f"Actual results folder: {save_dir}")

shutil.make_archive("/kaggle/working/taco_results", 'zip', save_dir)
print("Archive ready: /kaggle/working/taco_results.zip")

Actual results folder: /kaggle/working/runs_taco/yolo26n_taco
Archive ready: /kaggle/working/taco_results.zip
